# **The Master Joiner: Orbital Risk Synthesis**

**Datasets:** * **The Body:** `satcat_cleaned.csv` (Global Physical Registry - 67k+ Objects)
* **The Brain:** `ucs_cleaned.csv` (Active Intelligence Layer - 7.5k+ Payloads)

**Objective:** Fuse physical tracking data with operational intelligence to create a unified "Kinetic Master" registry for the 2026 Kessler Syndrome simulation.

### **The Engineering Challenge**
We currently possess two distinct realities: the **Physical Reality** (where objects are located) and the **Operational Reality** (what objects are doing). Merging these creates a significant "Visibility Gap"—we track ~67,000 objects, but only possess deep intelligence on ~11% (the active payload fleet).

To build a valid risk model, we must implement a synthesis pipeline:
1.  **Intelligence Coupling:** Perform a prioritized **Left Join** to enrich active assets without discarding the critical debris population.
2.  **Zombie Identification:** Algorithmically identify "The Living Dead"—payloads that are physically intact but operationally defunct (e.g., Age > Design Life).
3.  **Kinetic Engineering:** Calculate orbital velocity ($v$) and kinetic energy ($E_k = \frac{1}{2}mv^2$) for every object using derived mass and fuel-fraction logic.
4.  **Risk Synthesis:** Generate `kinetic_master.csv`, the single source of truth for geopolitical and collisional risk analysis.

In [1]:
import pandas as pd
import numpy as np
import utility as utils
from IPython.display import Markdown, display

### **Stage 1: Intelligence Coupling (The Master Merge)**
**The Problem:** We possess two disconnected datasets. The SATCAT contains the global population (including 40,000+ debris shards) but lacks mission context. The UCS registry contains deep mission intelligence but only for active payloads. Merging them blindly would duplicate mass columns and potentially discard the debris population if an inner join were used.

**The Solution:** 
* **Lean Selection:** We isolate only the "Intelligence Packet" from the UCS (metadata, mission, and lifecycle columns) to prevent column duplication.
* **Left Join Topology:** We utilize the SATCAT as the immutable backbone. This ensures that 100% of the debris and rocket bodies are retained in the final model.
* **Active Purge:** Post-merge, we immediately filter for `in_orbit == 1`, discarding decayed objects to focus the model strictly on the "Live Fire" environment of 2026.

In [2]:
# We enforce string types for IDs immediately to prevent "Invisible Bug" merge failures
print("Loading Gold Standard Registries...")
satcat = pd.read_csv('../data/clean/satcat_cleaned.csv', dtype={'norad_id': str, 'cospar_id': str}, low_memory=False)
ucs = pd.read_csv('../data/clean/ucs_cleaned.csv', dtype={'norad_id': str, 'cospar_id': str}, low_memory=False)

# Define the "Intelligence Packet"
# We strictly select only the metadata columns from UCS to avoid duplication.
# 'object_type' is NOT in this list because we rely on the SATCAT version.
ucs_intelligence_cols = [
    'norad_id',              # The Key
    'satellite_name',        # Human name
    'official_name',         # Full name
    'country_operator',      # Readable Country Name
    'users',                 # Sector string
    'primary_purpose',       # Standardized Mission (e.g., "Communications")
    'detailed_purpose',      # Granular Mission
    'orbit_type',            # Geometry label (e.g., "Polar")
    'is_commercial', 'is_government', 'is_military', 'is_civil', # Sector Flags
    'lifetime_years',        # Critical for Zombie Algorithm
    'power_watts',           # Extra Physics
    'un_registry',           # UN Registry Status
    'geo_longitude',         # GEO Slot (if applicable)
    'launch_site',           # Launch Site (if available)
    'contractor',            # Primary Contractor Name 
    'contractor_country'     # Contractor Country (if different from operator)
]

# Filter UCS down to just the intelligence packet
ucs_lean = ucs[ucs_intelligence_cols].copy()

# Execute Left Join
# SATCAT is the backbone (Left) so we keep all debris/rocket bodies
master = satcat.merge(ucs_lean, on='norad_id', how='left')

# Normalize merge collisions for duplicate field names
if 'launch_site_y' in master.columns or 'launch_site_x' in master.columns:
    master['launch_site'] = master.get('launch_site_y').combine_first(master.get('launch_site_x'))
    master = master.drop(columns=[c for c in ['launch_site_x', 'launch_site_y'] if c in master.columns])

if 'geo_longitude_y' in master.columns or 'geo_longitude_x' in master.columns:
    master['geo_longitude'] = master.get('geo_longitude_y').combine_first(master.get('geo_longitude_x'))
    master = master.drop(columns=[c for c in ['geo_longitude_x', 'geo_longitude_y'] if c in master.columns])

# The "Active Orbit" Purge
# We drop decayed objects to create the kinetic baseline
pre_purge_count = len(master)
master = master[master['in_orbit'] == 1].copy()
post_purge_count = len(master)

print(f"\n{'--- MERGE & PURGE AUDIT ---':^40}")
print(f"Global Registry (Raw):    {pre_purge_count:,}")
print(f"Active Kinetic Master:    {post_purge_count:,}")
print("-" * 40)
print(f"Intelligence Matches:     {master['satellite_name'].notna().sum():,} (Active Payloads)")

Loading Gold Standard Registries...



      --- MERGE & PURGE AUDIT ---       
Global Registry (Raw):    68,276
Active Kinetic Master:    33,358
----------------------------------------
Intelligence Matches:     5,505 (Active Payloads)


### **Stage 2: Zombie Identification & Schema Enforcement**
**The Problem:**
1.  **The "Living Dead":** A satellite labeled `OPERATIONAL` in the SATCAT might have launched in 1990 with a 5-year design life. This object is a "Zombie"—a high-mass threat that appears active in simple queries but is actually a piece of debris that cannot maneuver.
2.  **Visualization Blindness:** Our raw `object_type` lumps all payloads together, failing to distinguish between active assets and dead hulks for our risk dashboards.
3.  **Schema Chaos:** Merging two massive datasets has left our columns in a random order, making the dataset difficult for scientists to scan and audit.

**The Solution:**
* **Hybrid Health Algorithm:** We identify Zombies by cross-referencing `ops_status` (SATCAT) with `lifetime_years` (UCS). Any payload exceeding its design life by >10% is flagged.
* **Category Engineering:** We derive a high-level `category` column to satisfy downstream visualization needs (`Active Satellite`, `Inactive Satellite`, `Rocket Body`, `Debris`).
* **Scientific Reordering:** We reorganize the dataframe into strict engineering domains: **Identity**, **Kinetic Profile**, **Orbital State**, **Mission Intelligence**, and **Lifecycle**.

In [3]:
# Initialize the Zombie Flag (Default to 0)
master['is_zombie'] = 0

# Define Zombie Logic
# This only applies to payloads (satellites). Debris and rocket bodies are excluded 
# from zombie classification by design. The payload filter ensures we only flag inactive/non-operational satellites
# instead of snagging all dead objects regardless of type.
# Expected result: ~5,200-5,300 zombie payloads

payload_mask = master['object_type'] == 'PAYLOAD'
status_zombie = payload_mask & master['ops_status'].isin(['NON-OPERATIONAL', 'PARTIAL', 'STANDBY', 'DECAYED', 'UNKNOWN'])

# this is a filter created from a compound conditional statement
# ( is_payload AND ops_status is OPERATIONAL AND sat_age > ( lifetime + 10% )
lifecycle_zombie = (
    payload_mask & 
    (master['ops_status'] == 'OPERATIONAL') & 
    (master['sat_age_years'] > (master['lifetime_years'] * 1.1))
)

# now we set is_zomebie = 1 for any rows that match the compound filter: [status_zombie | lifecycle_zombie]
# df.loc[COMPOUND | CONDITION, COLUMN] = VALUE
master.loc[status_zombie | lifecycle_zombie, 'is_zombie'] = 1

master['category'] = master.apply(utils.derive_category, axis=1)

master['owner'] = master['owner_code']

# all were doing here is reordering columns into their logical groups
# to make visual inspection of the dataframe easier
logical_order = [
    # --- IDENTITY ---
    'norad_id', 'cospar_id', 
    'object_name', 
    'satellite_name', 'official_name', 'category', 'object_type',
    
    # --- KINETIC PROFILE ---
    'launch_mass_kg', 
    'proxy_mass_kg', 
    'dry_mass_kg', 'power_watts', 'rcs', 'rcs_class',
    
    # --- ORBITAL STATE ---
    'velocity_kms', 'kinetic_joules', 'semi_major_axis_km', 'proxy_power_watts',
    'orbit_class', 'orbit_type', 'period_minutes', 
    'perigee_km', 'apogee_km', 'inclination_degrees', 'eccentricity',
    
    # --- MISSION INTELLIGENCE ---
    'primary_purpose', 'detailed_purpose', 'users', 'country_operator',
    'is_commercial', 'is_government', 'is_military', 'is_civil',
    'geo_longitude', 'un_registry',
    
    # --- LIFECYCLE & STATUS ---
    'launch_date', 'launch_year', 'sat_age_years', 'lifetime_years',
    'launch_site',
    'ops_status', 
    'data_status', 
    'in_orbit', 'is_zombie',
    
    # --- SUPPLY CHAIN & METADATA ---
    'owner', 'owner_code', 'contractor', 'contractor_country'
]

# basically cross reference our logical_order column list with the actual columns in the master.
# as long as the column exists in the master, we keep it.

final_cols = [c for c in logical_order if c in master.columns]

# reassign master to itself but with the new column order. This doesn't drop any columns, just reorders them.
# I prefer C# and Javascript but I do like how flexible pandas is with this kind of thing. You can easily reorder, subset, 
# or even duplicate columns with simple list comprehensions.
master = master[final_cols]

# 5. Export
print(f"{'--- COMPATIBILITY AUDIT ---':^40}")
print(f"Object Name Density:      {master['object_name'].notna().mean():.1%} (Should be ~100%)")
print(f"Proxy Mass Present:       {'proxy_mass_kg' in master.columns}")
print(f"Final Schema Shape:       {master.shape[1]} columns")

output_path = '../data/clean/kinetic_master.csv'
master.to_csv(output_path, index=False)
print(f"\n✅ EXPORT SUCCESS: {len(master):,} active records saved to {output_path}")

      --- COMPATIBILITY AUDIT ---       
Object Name Density:      100.0% (Should be ~100%)
Proxy Mass Present:       True
Final Schema Shape:       43 columns



✅ EXPORT SUCCESS: 33,358 active records saved to ../data/clean/kinetic_master.csv


### **Stage 3: Kinetic & Power Engineering (The Physics Engine)**
**The Objective:** Transform static orbital elements into dynamic kinetic risk metrics and impute missing satellite bus capabilities.

**The Physics & Engineering Models:**
To quantify the "Stopping Power" and "Technological Density" of the debris field, we derive three critical variables:
1.  **Orbital Velocity ($v$):** Calculated using the standard gravitational parameter ($\mu$) and the semi-major axis ($a$) derived from the orbital period.
    * $$v = \sqrt{\frac{\mu}{a}}$$
2.  **Kinetic Energy ($E_k$):** The raw impact energy, measured in Joules.
    * $$E_k = \frac{1}{2} m v^2$$
3.  **Power Capacity ($P_{proxy}$):** A mass-derived estimate for power generation (Watts), using **Regime-Specific** densities (LEO vs. GEO) to respect the distinct physics of different orbital classes.
    * $$P_{proxy} \approx m \times \left( \frac{Watts}{kg} \right)_{orbit\_class}$$

**The Result:** A dataset that describes not just *where* objects are, but *how hard* they strike and *how capable* they were at launch.

In [4]:
# Define Astrodynamic Constants
MU = 398600.4418  # Standard Gravitational Parameter (km^3/s^2)

# Derive Semi-Major Axis (a) from Mean Motion
period_seconds = master['period_minutes'] * 60
mean_motion = (2 * np.pi) / period_seconds
master['semi_major_axis_km'] = np.cbrt(MU / mean_motion**2)

# Calculate Mean Orbital Velocity (km/s)
master['velocity_kms'] = np.sqrt(MU / master['semi_major_axis_km'])

# Calculate Kinetic Energy (Joules)
v_ms = master['velocity_kms'] * 1000
master['kinetic_joules'] = 0.5 * master['proxy_mass_kg'] * (v_ms ** 2)

# Initialize Proxy with Raw Data (Source of Truth)
master['proxy_power_watts'] = master['power_watts']

# Calculate "Tech Density" (Watts per Kg) for the KNOWN population
known_set = master.dropna(subset=['power_watts', 'proxy_mass_kg']).copy()
known_set['power_density'] = known_set['power_watts'] / known_set['proxy_mass_kg']

# Build the "Smart Lookup" Table (Median Density by Orbit Class)
# This captures the fact that GEO buses are fundamentally different from LEO cubesats.
orbit_density_map = known_set.groupby('orbit_class')['power_density'].median()
global_density = known_set['power_density'].median() # Fallback

print("--- SMART POWER MODEL PARAMETERS ---")
print(orbit_density_map)
print(f"Global Fallback: {global_density:.4f} W/kg")

# Apply the Smart Logic
master['proxy_power_watts'] = master.apply(utils.fill_power_smart, axis=1, args=(orbit_density_map, global_density))

# Final Schema Update
new_cols = ['velocity_kms', 'kinetic_joules', 'semi_major_axis_km', 'proxy_power_watts']
# Insert strictly into Kinetic Profile section
if 'rcs_class' in logical_order:
    insert_idx = logical_order.index('rcs_class') + 1
    final_schema = [c for c in logical_order if c not in new_cols]
    final_schema = final_schema[:insert_idx] + new_cols + final_schema[insert_idx:]
else:
    final_schema = master.columns.tolist()

master = master[[c for c in final_schema if c in master.columns]]

print(f"\n{'--- PHYSICS & ENGINEERING AUDIT ---':^50}")
print(f"Velocity Calculated:      {master['velocity_kms'].notna().mean():.1%}")
print(f"Kinetic Energy Calculated:{master['kinetic_joules'].notna().mean():.1%}")
print(f"Power Proxy Density:      {master['proxy_power_watts'].notna().mean():.1%} (Fully Filled for Known, Smart-Filled for Unknown)")

--- SMART POWER MODEL PARAMETERS ---
orbit_class
ELLIPTICAL    0.480519
GEO           2.181574
LEO           0.461538
MEO           1.081081
Name: power_density, dtype: float64
Global Fallback: 0.4615 W/kg



       --- PHYSICS & ENGINEERING AUDIT ---        
Velocity Calculated:      100.0%
Kinetic Energy Calculated:100.0%
Power Proxy Density:      100.0% (Fully Filled for Known, Smart-Filled for Unknown)


### **Stage 4: Export & Executive Completion Report**

**Objectives:**

- Generate a standardized summary using the reusable quick_report function.
- Export synthesized dataset.

**Why this approach?**

- The quick_report function provides a concise, readable, and consistent audit for any DataFrame or SQL table.
- By using quick_report, all data quality and schema checks are presented in a uniform format across the project.
- This approach is less over-engineered, easier to maintain, and ensures that all future audits (including SQL tables) will match the same reporting style.

**What does the report include?**
- Dataset dimensions and memory usage
- Duplicate primary key check (using norad_id)
- Data quality audit for all columns (nulls, fill %, type)
- Object and numeric column summaries

**Result:**

- The Kinetic Master Post-Synthesis Report below serves as the executive summary and final validation for the export.

**Output:**

- The synthesized dataset is saved as `kinetic_master.csv`.


In [5]:
utils.quick_report(master, title="Kinetic Master Post-Synthesis Report", key_col='norad_id')

output_path = '../data/clean/kinetic_master.csv'
master.to_csv(output_path, index=False)

# Read-back verification
export_check = pd.read_csv(output_path, dtype={'norad_id': str})
print(f"\n✅ EXPORT SUCCESS: {len(export_check):,} records saved to {output_path}")

# Kinetic Master Post-Synthesis Report

**Dimensions:** 33,358 rows × 47 columns

**Memory Footprint:** 40.49 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `object` | 0 | 100.0% | ✅ |
| **cospar_id** | `object` | 0 | 100.0% | ✅ |
| **object_name** | `object` | 0 | 100.0% | ✅ |
| **satellite_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **official_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **category** | `object` | 0 | 100.0% | ✅ |
| **object_type** | `object` | 0 | 100.0% | ✅ |
| **launch_mass_kg** | `float64` | 27,853 | 16.5% | ⚠️ |
| **proxy_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **dry_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **power_watts** | `float64` | 27,853 | 16.5% | ⚠️ |
| **rcs** | `float64` | 0 | 100.0% | ✅ |
| **rcs_class** | `object` | 0 | 100.0% | ✅ |
| **velocity_kms** | `float64` | 0 | 100.0% | ✅ |
| **kinetic_joules** | `float64` | 0 | 100.0% | ✅ |
| **semi_major_axis_km** | `float64` | 0 | 100.0% | ✅ |
| **proxy_power_watts** | `float64` | 0 | 100.0% | ✅ |
| **orbit_class** | `object` | 0 | 100.0% | ✅ |
| **orbit_type** | `object` | 27,853 | 16.5% | ⚠️ |
| **period_minutes** | `float64` | 0 | 100.0% | ✅ |
| **perigee_km** | `float64` | 0 | 100.0% | ✅ |
| **apogee_km** | `float64` | 0 | 100.0% | ✅ |
| **inclination_degrees** | `float64` | 0 | 100.0% | ✅ |
| **eccentricity** | `float64` | 0 | 100.0% | ✅ |
| **primary_purpose** | `object` | 27,853 | 16.5% | ⚠️ |
| **detailed_purpose** | `object` | 27,853 | 16.5% | ⚠️ |
| **users** | `object` | 27,853 | 16.5% | ⚠️ |
| **country_operator** | `object` | 27,853 | 16.5% | ⚠️ |
| **is_commercial** | `float64` | 27,853 | 16.5% | ⚠️ |
| **is_government** | `float64` | 27,853 | 16.5% | ⚠️ |
| **is_military** | `float64` | 27,853 | 16.5% | ⚠️ |
| **is_civil** | `float64` | 27,853 | 16.5% | ⚠️ |
| **geo_longitude** | `float64` | 0 | 100.0% | ✅ |
| **un_registry** | `object` | 27,853 | 16.5% | ⚠️ |
| **launch_date** | `object` | 0 | 100.0% | ✅ |
| **launch_year** | `int64` | 0 | 100.0% | ✅ |
| **sat_age_years** | `int64` | 0 | 100.0% | ✅ |
| **lifetime_years** | `float64` | 27,853 | 16.5% | ⚠️ |
| **launch_site** | `object` | 0 | 100.0% | ✅ |
| **ops_status** | `object` | 0 | 100.0% | ✅ |
| **data_status** | `object` | 32,414 | 2.8% | ⚠️ |
| **in_orbit** | `int64` | 0 | 100.0% | ✅ |
| **is_zombie** | `int64` | 0 | 100.0% | ✅ |
| **owner** | `object` | 0 | 100.0% | ✅ |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **contractor** | `object` | 27,853 | 16.5% | ⚠️ |
| **contractor_country** | `object` | 27,853 | 16.5% | ⚠️ |

### 📝 Object Overview
|                    |   count |   unique | top              |   freq |
|:-------------------|--------:|---------:|:-----------------|-------:|
| norad_id           |   33358 |    33358 | 5                |      1 |
| cospar_id          |   33358 |    33358 | 1958-002B        |      1 |
| object_name        |   33358 |    18858 | FENGYUN 1C DEB   |   2340 |
| satellite_name     |    5505 |     5492 | Starlink-2213    |      2 |
| official_name      |    5505 |     5483 | Jilin-1          |      5 |
| category           |   33358 |        5 | Active Satellite |  13106 |
| object_type        |   33358 |        4 | PAYLOAD          |  18339 |
| rcs_class          |   33358 |        4 | UNKNOWN          |  18639 |
| orbit_class        |   33358 |        5 | LEO              |  27302 |
| orbit_type         |    5505 |        5 | Inclined         |   2935 |
| primary_purpose    |    5505 |        6 | Communications   |   4329 |
| detailed_purpose   |    5505 |       43 | Not Specified    |   4799 |
| users              |    5505 |       15 | Commercial       |   4314 |
| country_operator   |    5505 |       89 | USA              |   3580 |
| un_registry        |    5505 |       61 | USA              |   3533 |
| launch_date        |   33358 |     3538 | 1999-05-10       |   2346 |
| launch_site        |   33358 |       63 | AFETR            |   7016 |
| ops_status         |   33358 |        7 | UNKNOWN          |  16567 |
| data_status        |     944 |        2 | NEA              |    943 |
| owner              |   33358 |      105 | US               |  17083 |
| owner_code         |   33358 |      105 | US               |  17083 |
| contractor         |    5505 |      370 | SpaceX           |   2963 |
| contractor_country |    5505 |       71 | USA              |   4087 |

### 📈 Numeric Overview
|                     |   count |            mean |             std |            min |            25% |            50% |            75% |              max |
|:--------------------|--------:|----------------:|----------------:|---------------:|---------------:|---------------:|---------------:|-----------------:|
| launch_mass_kg      |    5505 |   878.035       |  6251.37        |    1           |  227           |  260           |  290           | 450000           |
| proxy_mass_kg       |   33358 |   444.121       |  2591.27        |    1           |   50           |  290           |  355           | 450000           |
| dry_mass_kg         |   33358 |   382.169       |  2307.07        |    0.9         |   50           |  234           |  319.5         | 405000           |
| power_watts         |    5505 |  1228.67        |  3220.29        |    0           |  120           |  120           |  815           |  84000           |
| rcs                 |   33358 |     1.82635     |     9.444       |    0.0001      |    0.0181      |    1           |    1           |    830.035       |
| velocity_kms        |   33358 |     6.91903     |     1.36957     |    0.908319    |    7.25165     |    7.47771     |    7.62057     |      7.78233     |
| kinetic_joules      |   33358 |     8.87036e+09 |     7.36245e+10 |    2.81882e+07 |    1.38271e+09 |    7.49032e+09 |    1.03116e+10 |      1.31907e+13 |
| semi_major_axis_km  |   33358 | 10867.5         | 11481.3         | 6581.4         | 6863.77        | 7128.54        | 7579.91        | 483126           |
| proxy_power_watts   |   33358 |   281.143       |  1380.12        |    0           |    0           |  120           |  163.846       |  84000           |
| period_minutes      |   33358 |   240.796       |   709.497       |   88.56        |   94.32        |   99.83        |  109.46        |  55699.4         |
| perigee_km          |   33358 |  3160.82        |  9013.94        |  102           |  482           |  626           |  915           | 314973           |
| apogee_km           |   33358 |  5815.7         | 15931.1         |  206           |  487           |  786           | 1302           | 641287           |
| inclination_degrees |   33358 |    66.4617      |    28.2341      |    0           |   50           |   70           |   97.38        |    149.64        |
| eccentricity        |   33358 |     0.0564723   |     0.167341    |   -0.725044    |    0.00014611  |    0.00130404  |    0.00885031  |      0.972663    |
| is_commercial       |    5505 |     0.815985    |     0.387531    |    0           |    1           |    1           |    1           |      1           |
| is_government       |    5505 |     0.115531    |     0.319691    |    0           |    0           |    0           |    0           |      1           |
| is_military         |    5505 |     0.0995459   |     0.299421    |    0           |    0           |    0           |    0           |      1           |
| is_civil            |    5505 |     0.0179837   |     0.132904    |    0           |    0           |    0           |    0           |      1           |
| geo_longitude       |   33358 |     0.388015    |    12.536       | -179.8         |    0           |    0           |    0           |    359           |
| launch_year         |   33358 |  2006.72        |    19.297       | 1958           | 1992           | 2015.5         | 2024           |   2026           |
| sat_age_years       |   33358 |    19.283       |    19.297       |    0           |    2           |   10.5         |   34           |     68           |
| lifetime_years      |    5505 |     5.63465     |     3.58569     |    0.25        |    4           |    4           |    5           |     30           |
| in_orbit            |   33358 |     1           |     0           |    1           |    1           |    1           |    1           |      1           |
| is_zombie           |   33358 |     0.156874    |     0.363687    |    0           |    0           |    0           |    0           |      1           |


✅ EXPORT SUCCESS: 33,358 records saved to ../data/clean/kinetic_master.csv


C:\Users\demon\AppData\Local\Temp\ipykernel_23080\537260224.py:7: DtypeWarning: Columns (3,4,18,24,25,26,27,33,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  export_check = pd.read_csv(output_path, dtype={'norad_id': str})
